# COVID Parcel Business Analysis - Final Project
## Group 7
### Team Members:
- **Abeez Maredia**
- **Urvish Nayak**
- **Ayman Karovadiya**

### Project Overview:
This project analyzes the impact of COVID-19 on a parcel delivery business. We examine 
delivery volumes, revenue trends, regional performance, and operational efficiency 
during the pandemic period. Our goal is to provide executive-level insights and 
actionable recommendations.

### Date: April 2026

# 1. Importing libraries

In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2. Loading Dataset

In [52]:
df = pd.read_csv('COVID_Parcel_Business.csv')
print("Columns in dataset:", df.columns)

Columns in dataset: Index(['FakeCustomerID', 'THE_YEAR', 'THE_WEEK', 'VOLUME'], dtype='str')


# 3. Finding Required Columns

In [53]:
year_col = None
cust_col = None
week_col = None
parcel_col = None

for col in df.columns:
    col_lower = col.lower().strip()

    if 'year' in col_lower:
        year_col = col
    elif 'cust' in col_lower or 'id' in col_lower:
        cust_col = col
    elif 'week' in col_lower:
        week_col = col
    elif 'volume' in col_lower or 'parcel' in col_lower or 'vol' in col_lower:
        parcel_col = col

# Checking if all columns found
if year_col and cust_col and week_col and parcel_col:
    print("Columns detected successfully!")
else:
    print("Error: Could not detect required columns")

Columns detected successfully!


# 4. Defining Business Rules

In [54]:
BASE_COST = 22.0

SEGMENT_DEFS = {
    'Enterprise': {'min': 500001, 'max': float('inf'), 'discount': 0.22},
    'Large': {'min': 200001, 'max': 500000, 'discount': 0.17},
    'Medium': {'min': 10001, 'max': 200000, 'discount': 0.10},
    'Small': {'min': 1000, 'max': 10000, 'discount': 0.04}
}

# 5. Creating Baseline Data

In [55]:
df_pre_covid = df[
    (df[year_col] == 2019) |
    ((df[year_col] == 2020) & (df[week_col] <= 15))
]

df_baseline = df_pre_covid.groupby(cust_col)[parcel_col].sum().reset_index()
df_baseline.rename(columns={parcel_col: 'Vol_Baseline'}, inplace=True)

# 6. Assigning Customer Segment

In [56]:
def get_segment(volume):
    for seg in SEGMENT_DEFS:
        min_val = SEGMENT_DEFS[seg]['min']
        max_val = SEGMENT_DEFS[seg]['max']

        if volume >= min_val and volume <= max_val:
            return seg

    return 'Under 1K/New'

df_baseline['Segment'] = df_baseline['Vol_Baseline'].apply(get_segment)

# Merging segments back into main dataframe
df = pd.merge(df, df_baseline[[cust_col, 'Segment']], on=cust_col, how='left')
df['Segment'] = df['Segment'].fillna('Under 1K/New')

# 7. Calculating Revenue

In [57]:
def calculate_revenue(row):
    discount = 0

    for seg in SEGMENT_DEFS:
        if row['Segment'] == seg:
            discount = SEGMENT_DEFS[seg]['discount']

    revenue = row[parcel_col] * BASE_COST * (1 - discount)
    return revenue

df['Revenue'] = df.apply(calculate_revenue, axis=1)

# 8. Calculating Industry Growth Rate (ISGR)

In [58]:
vol_2019_pre = df[
    (df[year_col] == 2019) &
    (df[week_col] <= 15)
][parcel_col].sum()

vol_2020_pre = df[
    (df[year_col] == 2020) &
    (df[week_col] <= 15)
][parcel_col].sum()

isgr = ((vol_2020_pre - vol_2019_pre) / vol_2019_pre) * 100

print("ISGR:", round(isgr, 2), "%")

ISGR: 11.4 %


# 9. Calculating Peak Season Growth

In [59]:
peak_2019 = df[
    (df[year_col] == 2019) &
    (df[week_col] >= 45)
][parcel_col].sum()

peak_2020 = df[
    (df[year_col] == 2020) &
    (df[week_col] >= 45)
][parcel_col].sum()

peak_growth = ((peak_2020 - peak_2019) / peak_2019) * 100

print("Peak Growth:", round(peak_growth, 2), "%")

Peak Growth: 28.84 %
